# Выбор модели

In [9]:
import sys

from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import config

In [10]:
from init_pipeline import init_experiment
init_experiment(project_root, config)

Инициализация пропущена: /Users/romansafronenkov/Documents/Projects/uplift_modeling_pipeline/artifacts/boosting_pipeline/status.json уже существует и init=True


## Логирование

In [11]:
import os

from src.utils.logger import setup_logging

LOG_DIR = project_root / 'artifacts' / config.general.experiment_name / 'log'
os.makedirs(LOG_DIR, exist_ok=True)

LOG_FILE = LOG_DIR / 'choose_model.txt'

_logger = setup_logging(LOG_FILE, 'choose_model')

## Импорты

In [12]:
# ждем пока выполнится предыдущий ноутбук
import json
import time

while True:
    with open(project_root / 'artifacts' / config.general.experiment_name / 'status.json', 'r') as f:
        status = json.load(f)
    if status['features']:
        _logger.info('Признаки отобраны, выбираем модель')
        break
    time.sleep(60) 

[2026-09-05 10:04:51,655] - [choose_model] - [INFO] - Признаки отобраны, выбираем модель


In [13]:
import shutil
import copy
import json
import random
import time
from functools import partial

import joblib

from IPython.display import clear_output, display

import pandas as pd
import numpy as np

import optuna

from sklearn.model_selection import train_test_split, StratifiedKFold

from catboost import CatBoostRegressor, CatBoostClassifier
from sklearn.metrics import roc_auc_score, classification_report

from category_encoders import TargetEncoder
from sklearn.preprocessing import OrdinalEncoder

from sklift.models import SoloModel, TwoModels
from causalml.inference.meta import BaseXClassifier
from causalml.propensity import compute_propensity_score

from sklift.metrics import qini_auc_score, uplift_at_k

import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [14]:
from src.preprocessing import preprocess_sdf
from src.utils.uplift_clf_metrics import show_results
from src.utils.optimization import metric_stability

In [15]:
if config.general.load_venv_to_spark:
    os.environ['PYSPARK_PYTHON'] = './environment/bin/python'
    os.environ['PYSPARK_DRIVER_PYTHON'] = './environment/bin/python'

In [16]:
np.random.seed(config.general.seed)
random.seed(config.general.seed)

In [17]:
from src.utils.get_spark import get_conf, get_spark

conf = get_conf()

if config.general.load_venv_to_spark:
    conf.set('spark.archives', project_root / 'venv.tar.gz#environment')

In [18]:
spark = get_spark(conf, app_name=config.general.spark_session_name)

clear_output()
spark

## Константы

In [25]:
_logger.info(f"Experiment name: {config.general.experiment_name}")
data_dir = project_root / 'data'
fs_path = project_root / 'artifacts' / config.general.experiment_name / 'feature_selection'
choose_model_path = project_root / 'artifacts' / config.general.experiment_name / 'choose_model'
os.makedirs(choose_model_path, exist_ok=True)

[2026-09-05 10:07:06,617] - [choose_model] - [INFO] - Experiment name: boosting_pipeline


In [20]:
if config.general.choose_model:
    with open(fs_path / 'features_selected_final.json', 'r') as f:
        features = json.load(f)
    _logger.info(f'Признаки загружены: {len(features)}')

[2026-09-05 10:05:08,219] - [choose_model] - [INFO] - Признаки загружены: 25


## Загрузка и предобработка данных

In [22]:
if config.general.choose_model:
    features_to_select = []
    for feature in features:
        if feature.endswith('_diff'):
            features_to_select.append(feature.replace('_diff', ''))
            continue
        if feature.endswith('_sum'):
            features_to_select.append(feature.replace('_sum', ''))
            continue
        features_to_select.append(feature)

In [26]:
if config.general.choose_model:
    _logger.info('Загружаем датасет')
    dataset = spark.read.parquet(str(data_dir / 'train_dataset.parquet'))
    dataset = dataset.select(features_to_select + [config.dataset.date_col, config.dataset.treatment_col, config.dataset.target_col])

    len_dataset = dataset.count()
    _logger.info(f'Dataset length: {len_dataset}, dataset columns: {len(dataset.columns)}')

    dataset, dt_features, bitmask_cols = preprocess_sdf(dataset, config)
    dataset = dataset.select(features+[config.dataset.date_col, config.dataset.treatment_col, config.dataset.target_col])

    def find_string_features(sdf):
        def find_str(pair):
            col, dtype = pair
            if col not in [config.dataset.date_col, config.dataset.target_col, config.dataset.treatment_col]:
                if dtype == 'string':
                    return col
        return [pair[0] for pair in list(filter(find_str, sdf.dtypes))]

    string_features = find_string_features(dataset)
    string_cols_to_encode = [col for col in features if col in string_features]
    _logger.info(f"Строковых признаков: {len(string_cols_to_encode)}")

    train_dataset = dataset.toPandas()

[2026-09-05 10:07:09,173] - [choose_model] - [INFO] - Загружаем датасет
[2026-09-05 10:07:11,215] - [choose_model] - [INFO] - Dataset length: 458976, dataset columns: 28
[2026-09-05 10:07:11,216] - [src.preprocessing] - [INFO] - Num of datetime features: 0
[2026-09-05 10:07:11,216] - [src.preprocessing] - [INFO] - Num of bitmask features: 0
[2026-09-05 10:07:11,259] - [src.preprocessing] - [INFO] - Num of decimal features: 0
[2026-09-05 10:07:11,259] - [src.preprocessing] - [INFO] - Unique datatypes in dataset: {'string', 'double', 'int'}
[2026-09-05 10:07:11,288] - [choose_model] - [INFO] - Строковых признаков: 0


26/09/05 10:07:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

In [27]:
if config.general.choose_model:
    display(train_dataset.groupby([config.dataset.date_col, config.dataset.treatment_col])[config.dataset.target_col].mean())

date        treatment
2025-01-01  0            0.179402
            1            0.228907
2025-02-01  0            0.179402
            1            0.228907
2025-03-01  0            0.179402
            1            0.228907
2025-04-01  0            0.179402
            1            0.228907
2025-05-01  0            0.179369
            1            0.228907
2025-06-01  0            0.179369
            1            0.228907
2025-07-01  0            0.179369
            1            0.228907
2025-08-01  0            0.179369
            1            0.228907
2025-09-01  0            0.179376
            1            0.228907
Name: target, dtype: float64

In [28]:
spark.stop()

## Выбор модели

In [ ]:
if config.general.choose_model:
    X, y = train_dataset[features+[config.dataset.treatment_col]], train_dataset[config.dataset.target_col]
    dates = train_dataset[config.dataset.date_col]
    y_strat = X[config.dataset.treatment_col].astype(str) + '_' + dates.astype(str) + '_' + y.astype(str)

    kfold = StratifiedKFold(n_splits=4, shuffle=True, random_state=config.general.seed)

    models_results = {}

    if config.optimization.metric_to_optimize.value == 'qini':
        metric_func = qini_auc_score
    elif config.optimization.metric_to_optimize.value == 'uplift_at_k':
        metric_func = partial(uplift_at_k, k=config.optimization.k, strategy='by_group')

    key_metric_name = config.optimization.metric_to_optimize.value
    if key_metric_name == 'uplift_at_k':
        key_metric_name = key_metric_name.replace('k', str(int(config.optimization.k*100)))

In [38]:
if config.general.choose_model:
    _logger.info('*'*30+'S-learner'+'*'*30)

    metrics = {
        'qini': [],
        'uplift_at_10': [],
        'uplift_at_30': [],
        key_metric_name+'_choose': []
    }

    def objective(trial):
        params = {
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
            # 'num_leaves': trial.suggest_int('num_leaves', 2, 256),
            'max_depth': trial.suggest_int('max_depth', 2, 10),
            'iterations': trial.suggest_int('iterations', 100, 1200),
            'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.5),
            'rsm': trial.suggest_float('rsm', 0.4, 1.0),
            # 'bagging_temperature': trial.suggest_float('bagging_temperature', 0., 1.0),  # for Bayessian bootstrap
            'subsample': trial.suggest_float('subsample', 0.2, 1.0),  # for Bernoulli bootstrap
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 250),
            'auto_class_weights': trial.suggest_categorical('auto_class_weights', ['Balanced', 'SqrtBalanced', None]),
            'bootstrap_type': "Bernoulli",
            'random_state': config.general.seed,
            'verbose': False
        }

        metrics = []

        for (train_ix, val_ix) in kfold.split(X, y_strat):
            x_train, y_train = X.loc[train_ix, :], y.loc[train_ix]
            x_val, y_val = X.loc[val_ix, :], y.loc[val_ix]

            if len(string_cols_to_encode):
                encoder = TargetEncoder()
                x_train[string_cols_to_encode] = encoder.fit_transform(x_train[string_cols_to_encode], y=y_train)
                x_val[string_cols_to_encode] = encoder.transform(x_val[string_cols_to_encode])

            model = SoloModel(estimator=CatBoostClassifier(**params))
            model.fit(x_train[features], y_train, x_train[config.dataset.treatment_col])

            uplift_train = model.predict(x_train[features])
            uplift_val = model.predict(x_val[features])

            if all(uplift_val == 0):
                raise optuna.TrialPruned()

            metric_train = metric_func(y_train.values, uplift_train.ravel(), x_train[config.dataset.treatment_col].values)
            metric_val = metric_func(y_val.values, uplift_val.ravel(), x_val[config.dataset.treatment_col].values)

            metric = metric_stability(metric_train, metric_val, 0.2)
            metrics.append(metric)

        _logger.info(f'Trial #{trial.number}. Metrics: {metrics} (mean={np.mean(metrics):.4f}). Params: {trial.params}')
        return np.mean(metrics)

    _logger.info('Optimizing S-learner')

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=30, n_jobs=4)

    params = study.best_params
    params['bootstrap_type'] = 'Bernoulli'
    params['random_state'] = config.general.seed
    params['verbose'] = False

    _logger.info('Параметры подобраны, обучаем')
    i = 0
    for train_ix, val_ix in kfold.split(X, y_strat):
        x_train, y_train = X.loc[train_ix, :], y.loc[train_ix]
        x_val, y_val = X.loc[val_ix, :], y.loc[val_ix]

        if len(string_cols_to_encode):
            encoder = TargetEncoder()
            x_train[string_cols_to_encode] = encoder.fit_transform(x_train[string_cols_to_encode], y=y_train)
            x_val[string_cols_to_encode] = encoder.transform(x_val[string_cols_to_encode])

        model = SoloModel(estimator=CatBoostClassifier(**params))
        model.fit(x_train[features], y_train, x_train[config.dataset.treatment_col])

        uplift_val = model.predict(x_val[features])
        uplift_metrics = show_results(
            y_val.ravel(),
            uplift_val.ravel(),
            x_val[config.dataset.treatment_col].values.ravel(),
            save=True,
            save_path=choose_model_path,
            save_prefix=f'slearner_{i}'
        )
        metrics['uplift_at_10'].append(uplift_metrics['uplift_at_10'][0])
        metrics['uplift_at_30'].append(uplift_metrics['uplift_at_30'][0])
        metrics[key_metric_name+'_choose'].append(metric_func(y_val.values, uplift_val.ravel(), x_val[config.dataset.treatment_col].values))
        metrics['qini'].append(uplift_metrics['qini'][0])

        i += 1

    models_results['slearner'] = metrics
    clear_output()

In [39]:
if config.general.choose_model:
    _logger.info('*'*30+'T-learner'+'*'*30)

    metrics = {
        'qini': [],
        'uplift_at_10': [],
        'uplift_at_30': [],
        key_metric_name+'_choose': []
    }

    def objective(trial):
        params_tr = {
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg_tr', 1e-8, 10.0, log=True),
            # 'num_leaves': trial.suggest_int('num_leaves_tr', 2, 256),
            'max_depth': trial.suggest_int('max_depth_tr', 2, 10),
            'iterations': trial.suggest_int('iterations_tr', 100, 1200),
            'learning_rate': trial.suggest_float('learning_rate_tr', 0.001, 0.5),
            'rsm': trial.suggest_float('rsm_tr', 0.4, 1.0),
            # 'bagging_temperature': trial.suggest_float('bagging_temperature_tr', 0., 1.0),  # for Bayessian bootstrap
            'subsample': trial.suggest_float('subsample_tr', 0.2, 1.0),  # for Bernoulli bootstrap
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf_tr', 1, 250),
            'auto_class_weights': trial.suggest_categorical('auto_class_weights_tr', ['Balanced', 'SqrtBalanced', None]),
            'bootstrap_type': "Bernoulli",
            'random_state': config.general.seed,
            'verbose': False
        }

        params_c = {
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg_c', 1e-8, 10.0, log=True),
            # 'num_leaves': trial.suggest_int('num_leaves_c', 2, 256),
            'max_depth': trial.suggest_int('max_depth_c', 2, 10),
            'iterations': trial.suggest_int('iterations_c', 100, 1200),
            'learning_rate': trial.suggest_float('learning_rate_c', 0.001, 0.5),
            'rsm': trial.suggest_float('rsm_c', 0.4, 1.0),
            # 'bagging_temperature': trial.suggest_float('bagging_temperature_c', 0., 1.0),  # for Bayessian bootstrap
            'subsample': trial.suggest_float('subsample_c', 0.2, 1.0),  # for Bernoulli bootstrap
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf_c', 1, 250),
            'auto_class_weights': trial.suggest_categorical('auto_class_weights_c', ['Balanced', 'SqrtBalanced', None]),
            'bootstrap_type': "Bernoulli",
            'random_state': config.general.seed,
            'verbose': False
        }

        metrics = []

        for (train_ix, val_ix) in kfold.split(X, y_strat):
            x_train, y_train = X.loc[train_ix, :], y.loc[train_ix]
            x_val, y_val = X.loc[val_ix, :], y.loc[val_ix]

            if len(string_cols_to_encode):
                encoder = TargetEncoder()
                x_train[string_cols_to_encode] = encoder.fit_transform(x_train[string_cols_to_encode], y=y_train)
                x_val[string_cols_to_encode] = encoder.transform(x_val[string_cols_to_encode])

            model = TwoModels(
                estimator_trmnt=CatBoostClassifier(**params_tr),
                estimator_ctrl=CatBoostClassifier(**params_c)
            )
            model.fit(x_train[features], y_train, x_train[config.dataset.treatment_col])

            uplift_train = model.predict(x_train[features])
            uplift_val = model.predict(x_val[features])

            if all(uplift_val == 0):
                raise optuna.TrialPruned()

            metric_train = metric_func(y_train.values, uplift_train.ravel(), x_train[config.dataset.treatment_col].values)
            metric_val = metric_func(y_val.values, uplift_val.ravel(), x_val[config.dataset.treatment_col].values)

            metric = metric_stability(metric_train, metric_val, 0.2)
            metrics.append(metric)
            
        _logger.info(f'Trial #{trial.number}. Metrics: {metrics} (mean={np.mean(metrics):.4f}). Params: {trial.params}')
        return np.mean(metrics)

    _logger.info('Optimizing T-learner')

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=30, n_jobs=4)

    params = study.best_params
    params_tr = {'bootstrap_type': 'Bernoulli', 'random_state': config.general.seed, 'verbose': False}
    params_c = {'bootstrap_type': 'Bernoulli', 'random_state': config.general.seed, 'verbose': False}

    for key, value in params.items():
        if key.endswith('_c'):
            params_c[key[:-2]] = value
        elif key.endswith('_tr'):
            params_tr[key[:-3]] = value

    _logger.info('Параметры подобраны, обучаем')
    i = 0
    for train_ix, val_ix in kfold.split(X, y_strat):
        x_train, y_train = X.loc[train_ix, :], y.loc[train_ix]
        x_val, y_val = X.loc[val_ix, :], y.loc[val_ix]

        if len(string_cols_to_encode):
            encoder = TargetEncoder()
            x_train[string_cols_to_encode] = encoder.fit_transform(x_train[string_cols_to_encode], y=y_train)
            x_val[string_cols_to_encode] = encoder.transform(x_val[string_cols_to_encode])

        model = TwoModels(
                estimator_trmnt=CatBoostClassifier(**params_tr),
                estimator_ctrl=CatBoostClassifier(**params_c)
            )
        model.fit(x_train[features], y_train, x_train[config.dataset.treatment_col])

        uplift_val = model.predict(x_val[features])
        uplift_metrics = show_results(
            y_val.ravel(),
            uplift_val.ravel(),
            x_val[config.dataset.treatment_col].values.ravel(),
            save=True,
            save_path=choose_model_path,
            save_prefix=f'tlearner_{i}'
        )
        metrics['uplift_at_10'].append(uplift_metrics['uplift_at_10'][0])
        metrics['uplift_at_30'].append(uplift_metrics['uplift_at_30'][0])
        metrics[key_metric_name+'_choose'].append(metric_func(y_val.values, uplift_val.ravel(), x_val[config.dataset.treatment_col].values))
        metrics['qini'].append(uplift_metrics['qini'][0])

        i += 1

    models_results['tlearner'] = metrics
    clear_output()

In [41]:
if config.general.choose_model:
    _logger.info('*'*30+'X-learner'+'*'*30)
    # обучать ли модель propensity, если данные сбалансированы, то не нужно
    FIT_PROPENSITY = not 0.485 <= X[config.dataset.treatment_col].mean() <= 0.515

    class DummyPropensityModel:
        def predict(self, x):
            return np.full(shape=(x.shape[0],), fill_value=0.5)

    metrics = {
        'qini': [],
        'uplift_at_10': [],
        'uplift_at_30': [],
        key_metric_name+'_choose': []
    }

    cv_idxs = []
    propensity_models = []

    for train_ix, val_ix in kfold.split(X, y_strat):
        cv_idxs.append((train_ix, val_ix))

        x_train, y_train = X.loc[train_ix, :], y.loc[train_ix]
        x_val, y_val = X.loc[val_ix, :], y.loc[val_ix]

        x_train = x_train.assign(**{config.dataset.target_col: y_train})
        x_val = x_val.assign(**{config.dataset.target_col: y_val})

        if len(string_cols_to_encode):
            encoder = TargetEncoder()
            x_train[string_cols_to_encode] = encoder.fit_transform(x_train[string_cols_to_encode], y_train)
            x_val[string_cols_to_encode] = encoder.transform(x_val[string_cols_to_encode])

        (
            x_train
            .groupby(config.dataset.treatment_col)[config.dataset.target_col]
            .value_counts(normalize=True)
            .to_csv(choose_model_path / 'xlearner_x_train_stats.csv')
        )
        x_train[config.dataset.treatment_col].value_counts(normalize=True).to_csv(choose_model_path / 'xlearner_x_train_treatment.csv')

        (
            x_val
            .groupby(config.dataset.treatment_col)[config.dataset.target_col]
            .value_counts(normalize=True)
            .to_csv(choose_model_path / 'xlearner_x_val_stats.csv')
        )
        x_val[config.dataset.treatment_col].value_counts(normalize=True).to_csv(choose_model_path / 'xlearner_x_val_treatment.csv')

        if FIT_PROPENSITY:
            p, propensity_model = compute_propensity_score(
                x_train[features].values,
                x_train[config.dataset.treatment_col].values,
                calibrate_p=False
            )
        else:
            propensity_model = DummyPropensityModel()

        propensity_models.append(propensity_model)

    def objective(trial):
        params_co = {
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg_co', 1e-8, 10.0, log=True),
            # 'num_leaves': trial.suggest_int('num_leaves_co', 2, 256),
            'max_depth': trial.suggest_int('max_depth_co', 2, 10),
            'iterations': trial.suggest_int('iterations_co', 100, 1200),
            'learning_rate': trial.suggest_float('learning_rate_co', 0.001, 0.5),
            'rsm': trial.suggest_float('rsm_co', 0.4, 1.0),
            # 'bagging_temperature': trial.suggest_float('bagging_temperature_co', 0., 1.0),  # for Bayessian bootstrap
            'subsample': trial.suggest_float('subsample_co', 0.2, 1.0),  # for Bernoulli bootstrap
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf_co', 1, 250),
            'auto_class_weights': trial.suggest_categorical('auto_class_weights_co', ['Balanced', 'SqrtBalanced', None]),
            'bootstrap_type': "Bernoulli",
            'random_state': config.general.seed,
            'verbose': False
        }

        params_to = {
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg_to', 1e-8, 10.0, log=True),
            # 'num_leaves': trial.suggest_int('num_leaves_to', 2, 256),
            'max_depth': trial.suggest_int('max_depth_to', 2, 10),
            'iterations': trial.suggest_int('iterations_to', 100, 1200),
            'learning_rate': trial.suggest_float('learning_rate_to', 0.001, 0.5),
            'rsm': trial.suggest_float('rsm_to', 0.4, 1.0),
            # 'bagging_temperature': trial.suggest_float('bagging_temperature_to', 0., 1.0),  # for Bayessian bootstrap
            'subsample': trial.suggest_float('subsample_to', 0.2, 1.0),  # for Bernoulli bootstrap
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf_to', 1, 250),
            'auto_class_weights': trial.suggest_categorical('auto_class_weights_to', ['Balanced', 'SqrtBalanced', None]),
            'bootstrap_type': "Bernoulli",
            'random_state': config.general.seed,
            'verbose': False
        }

        params_ce = {
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg_ce', 1e-8, 10.0, log=True),
            # 'num_leaves': trial.suggest_int('num_leaves_ce', 2, 256),
            'loss_function': trial.suggest_categorical('loss_function_ce', ['RMSE', 'MAE']),
            'max_depth': trial.suggest_int('max_depth_ce', 2, 10),
            'iterations': trial.suggest_int('iterations_ce', 100, 1200),
            'learning_rate': trial.suggest_float('learning_rate_ce', 0.001, 0.5),
            'rsm': trial.suggest_float('rsm_ce', 0.4, 1.0),
            # 'bagging_temperature': trial.suggest_float('bagging_temperature_ce', 0., 1.0),  # for Bayessian bootstrap
            'subsample': trial.suggest_float('subsample_ce', 0.2, 1.0),  # for Bernoulli bootstrap
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf_ce', 1, 250),
            'bootstrap_type': "Bernoulli",
            'random_state': config.general.seed,
            'verbose': False
        }

        params_te = {
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg_te', 1e-8, 10.0, log=True),
            # 'num_leaves': trial.suggest_int('num_leaves_te', 2, 256),
            'loss_function': trial.suggest_categorical('loss_function_te', ['RMSE', 'MAE']),
            'max_depth': trial.suggest_int('max_depth_te', 2, 10),
            'iterations': trial.suggest_int('iterations_te', 100, 1200),
            'learning_rate': trial.suggest_float('learning_rate_te', 0.001, 0.5),
            'rsm': trial.suggest_float('rsm_te', 0.4, 1.0),
            # 'bagging_temperature': trial.suggest_float('bagging_temperature_te', 0., 1.0),  # for Bayessian bootstrap
            'subsample': trial.suggest_float('subsample_te', 0.2, 1.0),  # for Bernoulli bootstrap
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf_te', 1, 250),
            'bootstrap_type': "Bernoulli",
            'random_state': config.general.seed,
            'verbose': False
        }

        metrics = []

        for (train_ix, val_ix), propensity_model in zip(cv_idxs, propensity_models):
            x_train, y_train = X.loc[train_ix, :], y.loc[train_ix]
            x_val, y_val = X.loc[val_ix, :], y.loc[val_ix]

            if len(string_cols_to_encode):
                encoder = TargetEncoder()
                x_train[string_cols_to_encode] = encoder.fit_transform(x_train[string_cols_to_encode], y=y_train)
                x_val[string_cols_to_encode] = encoder.transform(x_val[string_cols_to_encode])

            model = BaseXClassifier(
                control_outcome_learner=CatBoostClassifier(**params_co),
                treatment_outcome_learner=CatBoostClassifier(**params_to),
                control_effect_learner=CatBoostRegressor(**params_ce),
                treatment_effect_learner=CatBoostRegressor(**params_te)
            )

            p = propensity_model.predict(x_train[features])
            
            model.fit(X=x_train[features].values, treatment=x_train[config.dataset.treatment_col], y=y_train.ravel(), p=p)

            uplift_train = model.predict(x_train[features].values, p=propensity_model.predict(x_train[features]))
            uplift_val = model.predict(x_val[features].values, p=propensity_model.predict(x_val[features]))

            if all(uplift_val == 0):
                raise optuna.TrialPruned()

            metric_train = metric_func(y_train.values, uplift_train.ravel(), x_train[config.dataset.treatment_col].values)
            metric_val = metric_func(y_val.values, uplift_val.ravel(), x_val[config.dataset.treatment_col].values)

            metric = metric_stability(metric_train, metric_val, 0.2)
            metrics.append(metric)
            
        _logger.info(f'Trial #{trial.number}. Metrics: {metrics} (mean={np.mean(metrics):.4f}). Params: {trial.params}')
        return np.mean(metrics)

    _logger.info('Optimizing X-learner')

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=30, n_jobs=4)

    params = study.best_params
    params_co = {'bootstrap_type': 'Bernoulli', 'random_state': config.general.seed, 'verbose': False}
    params_to = {'bootstrap_type': 'Bernoulli', 'random_state': config.general.seed, 'verbose': False}
    params_ce = {'bootstrap_type': 'Bernoulli', 'random_state': config.general.seed, 'verbose': False}
    params_te = {'bootstrap_type': 'Bernoulli', 'random_state': config.general.seed, 'verbose': False}

    for key, value in params.items():
        if key.endswith('_co'):
            params_co[key[:-3]] = value
        elif key.endswith('_to'):
            params_to[key[:-3]] = value
        elif key.endswith('_ce'):
            params_ce[key[:-3]] = value
        elif key.endswith('_te'):
            params_te[key[:-3]] = value

    _logger.info('Параметры подобраны, обучаем')
    i = 0
    for train_ix, val_ix in kfold.split(X, y_strat):
        x_train, y_train = X.loc[train_ix, :], y.loc[train_ix]
        x_val, y_val = X.loc[val_ix, :], y.loc[val_ix]

        if len(string_cols_to_encode):
            encoder = TargetEncoder()
            x_train[string_cols_to_encode] = encoder.fit_transform(x_train[string_cols_to_encode], y=y_train)
            x_val[string_cols_to_encode] = encoder.transform(x_val[string_cols_to_encode])

        model = BaseXClassifier(
                control_outcome_learner=CatBoostClassifier(**params_co),
                treatment_outcome_learner=CatBoostClassifier(**params_to),
                control_effect_learner=CatBoostRegressor(**params_ce),
                treatment_effect_learner=CatBoostRegressor(**params_te)
            )

        if FIT_PROPENSITY:
            p, propensity_model = compute_propensity_score(
                x_train[features].values,
                x_train[config.dataset.treatment_col].values,
                calibrate_p=False)
        else:
            propensity_model = DummyPropensityModel()
            p = propensity_model.predict(x_train[features])
            
        model.fit(X=x_train[features].values, treatment=x_train[config.dataset.treatment_col], y=y_train.ravel(), p=p)

        uplift_val = model.predict(x_val[features].values, p=propensity_model.predict(x_val[features]))
        
        uplift_metrics = show_results(
            y_val.ravel(),
            uplift_val.ravel(),
            x_val[config.dataset.treatment_col].values.ravel(),
            save=True,
            save_path=choose_model_path,
            save_prefix=f'xlearner_{i}'
        )
        metrics['uplift_at_10'].append(uplift_metrics['uplift_at_10'][0])
        metrics['uplift_at_30'].append(uplift_metrics['uplift_at_30'][0])
        metrics[key_metric_name+'_choose'].append(metric_func(y_val.values, uplift_val.ravel(), x_val[config.dataset.treatment_col].values))
        metrics['qini'].append(uplift_metrics['qini'][0])

        i += 1

    models_results['xlearner'] = metrics
    clear_output()

In [42]:
if config.general.choose_model:
    mean_stats = {}

    for key in models_results.keys():
        mean_stats[key] = {}
        for metric in models_results[key].keys():
            mean_stats[key][metric] = np.mean(models_results[key][metric])
    _logger.info(f"Model choosing mean stats:\n{mean_stats}")

[2026-09-05 11:41:55,504] - [choose_model] - [INFO] - Model choosing mean stats:
{'slearner': {'qini': 0.23484820294194503, 'uplift_at_10': 0.40318646485049126, 'uplift_at_30': 0.20923299482597418, 'qini_choose': 0.23484820294194503}, 'tlearner': {'qini': 0.21077785977518645, 'uplift_at_10': 0.3371365279256418, 'uplift_at_30': 0.19631922872499882, 'qini_choose': 0.21077785977518645}, 'xlearner': {'qini': 0.23283782375004047, 'uplift_at_10': 0.399653466459576, 'uplift_at_30': 0.21216635898794564, 'qini_choose': 0.23283782375004047}}


In [45]:
if config.general.choose_model:
    mean_stats = pd.DataFrame(mean_stats)
    choosen_model = mean_stats.loc[key_metric_name+'_choose'][mean_stats.loc[key_metric_name] == mean_stats.loc[key_metric_name].max()].index[0]
    with open(choose_model_path / 'choosen_model.json', 'w') as f:
        json.dump(choosen_model, f)

    _logger.info(f'Choosen model is {choosen_model}')

[2026-09-05 11:51:54,399] - [choose_model] - [INFO] - Choosen model is slearner


In [46]:
with open(project_root / 'artifacts' / config.general.experiment_name / 'status.json', 'w') as f:
    status = {
        'init': True,
        'dataset': True,
        'features': True,
        'choose_model': True,
        'optimization': False,
        'fitting': False
    }
    json.dump(status, f)

In [ ]:
os._exit(00)